In [ ]:
import pandas as pd


**ببساطة:** المفروض تشوف أول 5 صفوف من الداتا. راجع أسماء الأعمدة والقيم؛ دي عينة صغيرة للتأكد إن الملف اتقري صح.

In [ ]:
import os
import pandas as pd

# Load the CSV from the project folder (works in Colab and locally if the file is beside this notebook)
csv_path = 'Olist_Master_Dataset.csv'
if not os.path.exists(csv_path):
    # Optional Google Colab upload fallback
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = next(iter(uploaded))
    except Exception:
        raise FileNotFoundError('Put Olist_Master_Dataset.csv in the same folder as this notebook.')

df = pd.read_csv(csv_path)
df.head()

In [ ]:
# Keep a copy before any cleaning so we can measure the impact later
df_original = df.copy(deep=True)
print("Original dataset copy saved:", df_original.shape)


**ببساطة:** `head()` بيعرض أول 5 صفوف عشان نتأكد إن ملف الداتا اتقري صح. راجع أسماء الأعمدة والقيم اللي ظهرت.


---

# EDA — Data Size and Structure

الجزء ده بيبدأ بفهم حجم الداتا وأنواع الأعمدة قبل ما نستنتج أي حاجة. التحليلات هنا بتستخدم نسخة الداتا الأصلية قبل التنظيف.


In [ ]:
print("df.shape =", df_original.shape)
print("Rows:", df_original.shape[0])
print("Columns:", df_original.shape[1])

**ببساطة:** `shape` بتعرض (عدد الصفوف، عدد الأعمدة). الأرقام اللي هتظهر عندك هي الحجم الفعلي للداتا الأصلية.


In [ ]:
df_original.info()

**ببساطة:** `info()` بتوضح نوع كل عمود وعدد القيم غير الفاضية. لو Non-Null Count أقل من عدد الصفوف، يبقى العمود فيه قيم ناقصة.


In [ ]:
display(df_original.describe(include="all").T)

**ببساطة:** `describe()` بتلخص الأعمدة الرقمية والنصية. للأرقام ركّز على المتوسط والوسيط (50%) وأقل وأعلى قيمة؛ وللنصوص ركّز على unique وtop وfreq.


In [ ]:
missing_summary = pd.DataFrame({"Missing Values": df_original.isna().sum(), "Missing %": (df_original.isna().mean()*100).round(2)})
missing_summary = missing_summary[missing_summary["Missing Values"] > 0].sort_values("Missing Values", ascending=False)
display(missing_summary if not missing_summary.empty else pd.DataFrame({"Result":["No missing values found"]}))

**ببساطة:** كل صف في الجدول عمود فيه نقص؛ `Missing %` هي نسبة النقص من إجمالي الصفوف. لو الجدول فاضي فمعناه إن مفيش قيم ناقصة.


In [ ]:
print("Exact duplicate rows:", int(df_original.duplicated().sum()))
print("Rows before duplicate removal:", len(df_original))
print("Rows after duplicate removal (theoretical):", len(df_original.drop_duplicates()))

**ببساطة:** ده بيحسب الصفوف المتطابقة بالكامل فقط، مش تكرار `order_id` لأن الطلب الواحد ممكن يظهر في أكتر من صف بسبب المنتجات.



---

# Univariate Analysis

تحليل كل متغير لوحده: توزيع الأرقام وأكتر القيم النصية تكرارًا.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

eda_df = df_original.copy()
for col in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date", "review_creation_date", "review_answer_timestamp"]:
    if col in eda_df.columns:
        eda_df[col] = pd.to_datetime(eda_df[col], errors="coerce")

numeric_cols = eda_df.select_dtypes(include="number").columns.tolist()
print("Numeric columns:", numeric_cols)
plot_cols = numeric_cols[:12]
if plot_cols:
    eda_df[plot_cols].hist(figsize=(16, 12), bins=30, edgecolor="black")
    plt.suptitle("Univariate Analysis — Numeric Features")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns found.")

**ببساطة:** كل رسم بيعرض عمود رقمي لوحده. لو التوزيع مائل ناحية اليمين، غالبًا فيه قيم كبيرة قليلة؛ وده شائع في السعر والشحن.


In [ ]:
cat_cols = [c for c in ["order_status", "payment_type", "customer_state", "product_category_name_english", "product_category_name"] if c in eda_df.columns]
for col in cat_cols:
    counts = eda_df[col].fillna("Missing").astype(str).value_counts().head(10)
    display(counts.rename_axis(col).reset_index(name="Count"))
    plt.figure(figsize=(10, 4))
    sns.barplot(x=counts.values, y=counts.index)
    plt.title(f"Top values — {col}")
    plt.xlabel("Count")
    plt.tight_layout()
    plt.show()

**ببساطة:** كل جدول بيعرض أكتر 10 قيم تكرارًا في العمود. أعلى Count معناها القيمة الأكثر ظهورًا، مش بالضرورة الأعلى مبيعًا.



---

# Boxplots for Features

الـ Boxplot بيساعدنا نلاحظ القيم المتطرفة في المتغيرات الرقمية.


In [ ]:
box_cols = [c for c in ["price", "freight_value", "payment_value", "payment_installments", "review_score", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"] if c in eda_df.columns and pd.api.types.is_numeric_dtype(eda_df[c])]
for col in box_cols:
    plt.figure(figsize=(9, 3))
    sns.boxplot(x=eda_df[col].dropna())
    plt.title(f"Boxplot — {col}")
    plt.tight_layout()
    plt.show()
    q1, q3 = eda_df[col].quantile([0.25, 0.75])
    iqr = q3-q1
    out_count = int(((eda_df[col] < q1-1.5*iqr) | (eda_df[col] > q3+1.5*iqr)).sum())
    print(f"{col}: Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, potential outliers={out_count}")

**ببساطة:** النقط أو الأطراف البعيدة عن الصندوق تعتبر قيمًا متطرفة محتملة حسب قاعدة IQR. مش لازم نحذفها تلقائيًا؛ ممكن تكون طلبات حقيقية غالية.



---

# Bivariate Analysis

هنا بنقارن متغيرين مع بعض عشان نشوف الفرق أو العلاقة بينهم.


In [ ]:
if "payment_type" in eda_df.columns and "payment_value" in eda_df.columns:
    payment_compare = eda_df.groupby("payment_type", dropna=False)["payment_value"].agg(["count", "mean", "median", "sum"]).sort_values("sum", ascending=False)
    display(payment_compare.round(2))
    payment_compare["sum"].sort_values(ascending=False).plot(kind="bar", figsize=(9,4), title="Total Payment Value by Payment Type")
    plt.ylabel("Total payment value")
    plt.tight_layout(); plt.show()
else:
    print("Required columns are not available.")

**ببساطة:** الجدول بيقارن طرق الدفع من حيث عدد السجلات ومتوسط وقيمة المدفوعات الإجمالية. أكبر Sum معناها أكبر إجمالي قيمة مسجلة لطريقة الدفع.


In [ ]:
if "customer_state" in eda_df.columns and "price" in eda_df.columns:
    state_price = eda_df.groupby("customer_state").agg(orders=("order_id", "nunique") if "order_id" in eda_df.columns else ("price", "size"), total_price=("price", "sum"), avg_price=("price", "mean")).sort_values("total_price", ascending=False).head(15)
    display(state_price.round(2))
    state_price["total_price"].sort_values().plot(kind="barh", figsize=(9,6), title="Top States by Total Product Price")
    plt.tight_layout(); plt.show()
else:
    print("Required columns are not available.")

**ببساطة:** بنقارن الولايات من حيث إجمالي السعر ومتوسطه وعدد الطلبات الفريدة لو عمود الطلب موجود. إجمالي السعر مش هو نفسه عدد العملاء.


In [ ]:
if "review_score" in eda_df.columns and "price" in eda_df.columns:
    display(eda_df.groupby("review_score", dropna=False).agg(records=("price", "size"), avg_price=("price", "mean"), median_price=("price", "median")).round(2))
    sns.boxplot(data=eda_df, x="review_score", y="price")
    plt.title("Product Price by Review Score")
    plt.tight_layout(); plt.show()
else:
    print("review_score or price column is unavailable.")

**ببساطة:** ده بيقارن السعر بتقييم العميل. وجود فرق بين متوسطات الأسعار لا يثبت إن السعر هو سبب التقييم؛ دي علاقة وصفية فقط.



---

# Multivariate Analysis

التحليل متعدد المتغيرات بيجمع أكتر من متغير في نفس المقارنة.


In [ ]:
multi_cols = [c for c in ["price", "freight_value", "payment_value", "review_score", "payment_installments", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"] if c in eda_df.columns and pd.api.types.is_numeric_dtype(eda_df[c])]
if len(multi_cols) >= 2:
    sample = eda_df[multi_cols].sample(min(5000, len(eda_df)), random_state=42)
    sns.pairplot(sample, corner=True, diag_kind="hist")
    plt.show()
else:
    print("Not enough numeric columns for pairplot.")

**ببساطة:** كل نقطة في الرسومات بتساعدنا نقارن متغيرين، والقطر بيعرض توزيع كل متغير. استخدم عينة محدودة لتقليل وقت الرسم.


In [ ]:
if len(multi_cols) >= 2:
    corr = eda_df[multi_cols].corr(numeric_only=True)
    plt.figure(figsize=(11, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Correlation Matrix — Numeric Features")
    plt.tight_layout(); plt.show()
    display(corr.round(3))
else:
    print("Not enough numeric columns for correlation matrix.")

**ببساطة:** كل خلية بتوضح ارتباط متغيرين. +1 ارتباط موجب قوي، و-1 سالب قوي، والصفر تقريبًا ارتباط خطي ضعيف. الارتباط مش معناه سببية.



---

# Correlation Analysis

دراسة الارتباطات بين المتغيرات الرقمية بالأرقام وبالرسم.


In [ ]:
if len(multi_cols) >= 2:
    corr_pairs = corr.where(~__import__("numpy").triu(__import__("numpy").ones(corr.shape), k=1).astype(bool)).stack().reset_index()
    corr_pairs.columns = ["Feature 1", "Feature 2", "Correlation"]
    corr_pairs["Absolute Correlation"] = corr_pairs["Correlation"].abs()
    display(corr_pairs.sort_values("Absolute Correlation", ascending=False).head(15).round(3))
else:
    print("Not enough numeric columns.")

**ببساطة:** الجدول بيرتب أقوى الارتباطات بالقيمة المطلقة. راجع أسماء العمودين واتجاه الارتباط، وخلي بالك إن العلاقة مش دليل على سبب ونتيجة.



---

# Sales & Categories

تحليل المبيعات الشهرية والفئات الأعلى من حيث قيمة المنتجات.


In [ ]:
sales_df = eda_df.copy()
if "order_purchase_timestamp" in sales_df.columns and "price" in sales_df.columns:
    sales_df["year_month"] = sales_df["order_purchase_timestamp"].dt.to_period("M").astype(str)
    monthly_sales = sales_df.groupby("year_month")["price"].sum().sort_index()
    display(monthly_sales.rename("Total Product Price").to_frame().tail(24).round(2))
    monthly_sales.tail(24).plot(figsize=(12,4), marker="o", title="Monthly Sales (sum of product price)")
    plt.ylabel("Total product price")
    plt.xticks(rotation=45); plt.tight_layout(); plt.show()
else:
    print("Purchase timestamp or price column is unavailable.")

**ببساطة:** الخط بيعرض مجموع `price` لكل شهر، وليس بالضرورة إجمالي المدفوعات أو الأرباح. أعلى نقطة هي الشهر صاحب أكبر مجموع أسعار منتجات.


In [ ]:
category_col = "product_category_name_english" if "product_category_name_english" in eda_df.columns else ("product_category_name" if "product_category_name" in eda_df.columns else None)
if category_col and "price" in eda_df.columns:
    category_sales = eda_df.groupby(category_col, dropna=False).agg(total_price=("price", "sum"), avg_price=("price", "mean"), records=("price", "size")).sort_values("total_price", ascending=False).head(15)
    display(category_sales.round(2))
    category_sales["total_price"].sort_values().plot(kind="barh", figsize=(10,6), title="Top Product Categories by Total Price")
    plt.tight_layout(); plt.show()
else:
    print("Category or price column is unavailable.")

**ببساطة:** الفئات مترتبة حسب مجموع أسعار المنتجات. عمود Records بيوضح حجم السجلات في كل فئة، وده مهم قبل مقارنة المتوسطات.



---

# Delivery Analysis

تحليل مدة التوصيل والتأخير مقارنة بموعد التسليم المتوقع.


In [ ]:
delivery = eda_df.copy()
needed_dates = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
if all(c in delivery.columns for c in needed_dates):
    delivery["delivery_days"] = (delivery["order_delivered_customer_date"] - delivery["order_purchase_timestamp"]).dt.total_seconds()/86400
    delivery["delay_days"] = (delivery["order_delivered_customer_date"] - delivery["order_estimated_delivery_date"]).dt.total_seconds()/86400
    delivery_valid = delivery[delivery["delivery_days"].notna() & (delivery["delivery_days"] >= 0)]
    print("Valid delivered records:", len(delivery_valid))
    display(delivery_valid[["delivery_days", "delay_days"]].describe().round(2))
    delivery_valid["delivery_days"].plot(kind="hist", bins=40, figsize=(9,4), title="Delivery Duration in Days")
    plt.xlabel("Days from purchase to delivery"); plt.tight_layout(); plt.show()
    late_pct = (delivery_valid["delay_days"] > 0).mean()*100
    print(f"Late delivery rate: {late_pct:.2f}%")
    print("On time or early:", int((delivery_valid["delay_days"] <= 0).sum()))
    print("Late:", int((delivery_valid["delay_days"] > 0).sum()))
else:
    print("Required delivery date columns are unavailable.")

**ببساطة:** مدة التوصيل هي الأيام بين الشراء والاستلام. التأخير موجب لو الاستلام بعد الموعد المتوقع، وصفر أو سالب لو في الموعد أو قبله. النسبة محسوبة على السجلات الصالحة فقط.



---

# Customer Reviews

تحليل تقييمات العملاء وتجربة الشراء، مع فحص التعليقات النصية إن كانت موجودة.


In [ ]:
if "review_score" in eda_df.columns:
    score_counts = eda_df["review_score"].value_counts(dropna=False).sort_index()
    display(score_counts.rename_axis("Review Score").to_frame("Count"))
    print("Average review score:", round(eda_df["review_score"].mean(), 3))
    print("Median review score:", eda_df["review_score"].median())
    score_counts.plot(kind="bar", figsize=(8,4), title="Customer Review Score Distribution")
    plt.xlabel("Review score"); plt.ylabel("Count"); plt.tight_layout(); plt.show()
else:
    print("review_score column is unavailable.")

for c in ["review_comment_title", "review_comment_message"]:
    if c in eda_df.columns:
        nonempty = eda_df[c].fillna("").astype(str).str.strip().ne("")
        print(f"{c}: non-empty={int(nonempty.sum()):,} / {len(eda_df):,} ({nonempty.mean()*100:.2f}%)")

**ببساطة:** الدرجات بتوضح التقييمات الأكثر شيوعًا ومتوسط ووسيط التقييم. نسبة التعليقات غير الفاضية بتوضح كام سجل فيه نص مكتوب، لكنها مش تحليل مشاعر.


In [ ]:
if "review_score" in eda_df.columns and "customer_state" in eda_df.columns:
    review_state = eda_df.groupby("customer_state")["review_score"].agg(["count", "mean", "median"]).sort_values("count", ascending=False).head(15)
    display(review_state.round(2))
else:
    print("review_score or customer_state is unavailable.")

**ببساطة:** الجدول بيقارن متوسط التقييم حسب الولاية، مع عدد السجلات عشان ما نعتمدش على متوسط ناتج من عدد صغير جدًا من التقييمات.



---

# Geographic Analysis

مقارنة الولايات من حيث عدد الطلبات والعملاء والقيمة الإجمالية.


In [ ]:
if "customer_state" in eda_df.columns:
    agg = {}
    if "order_id" in eda_df.columns: agg["orders"] = ("order_id", "nunique")
    if "customer_unique_id" in eda_df.columns: agg["unique_customers"] = ("customer_unique_id", "nunique")
    if "price" in eda_df.columns: agg["total_product_price"] = ("price", "sum")
    if "review_score" in eda_df.columns: agg["avg_review_score"] = ("review_score", "mean")
    if agg:
        geo = eda_df.groupby("customer_state").agg(**agg).sort_values(next(iter(agg.keys())), ascending=False)
        display(geo.head(20).round(2))
        geo.iloc[:,0].head(15).sort_values().plot(kind="barh", figsize=(9,6), title=f"Top States by {geo.columns[0]}")
        plt.tight_layout(); plt.show()
    else:
        print("No aggregation columns are available.")
else:
    print("customer_state column is unavailable.")

**ببساطة:** الجدول بيعرض الولايات حسب المقاييس المتاحة. عدد العملاء الفريدين مختلف عن عدد الطلبات، وإجمالي السعر لا يمثل الربح.


In [ ]:
if all(c in eda_df.columns for c in ["customer_lat", "customer_lng"]):
    geo_points = eda_df[["customer_lat", "customer_lng"]].dropna()
    if len(geo_points) > 10000: geo_points = geo_points.sample(10000, random_state=42)
    plt.figure(figsize=(8,6))
    plt.scatter(geo_points["customer_lng"], geo_points["customer_lat"], s=3, alpha=0.25)
    plt.xlabel("Longitude"); plt.ylabel("Latitude"); plt.title("Customer Geographic Distribution (sample)")
    plt.tight_layout(); plt.show()
    print("Points plotted:", len(geo_points))
else:
    print("Customer coordinates are unavailable.")

**ببساطة:** كل نقطة بتمثل موقع عميل حسب الإحداثيات المتاحة. الرسم عينة لو البيانات كبيرة، وبيوضح التوزيع الجغرافي بشكل عام مش حدود الولايات.


In [ ]:
date_cols = ['order_purchase_timestamp', 'order_approved_at', 
             'order_delivered_carrier_date', 'order_delivered_customer_date', 
             'order_estimated_delivery_date']

**ببساطة:** دي أسماء أعمدة التواريخ اللي هنحوّلها لنوع تاريخ علشان نقدر نحسب مدة التوصيل ونحلل الزمن.

In [ ]:
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')



**ببساطة:** هنا بنحوّل أعمدة الوقت لتاريخ/وقت. لو عمود مش موجود بيتخطاه الكود، ولو قيمة تاريخ غير مفهومة بتتحول لقيمة ناقصة.

In [ ]:


df.dropna(subset=['order_delivered_customer_date'], inplace=True)

if 'review_score' in df.columns:
    df['review_score'] = df['review_score'].fillna(df['review_score'].median())

if 'product_category_name_english' in df.columns:
    df['product_category_name_english'] = df['product_category_name_english'].fillna('unknown')

**ببساطة:** بننظّف بعض القيم الناقصة. الطلب اللي مفيش له تاريخ تسليم مش هينفع نستخدمه في تحليل مدة التوصيل. راجع عدد الصفوف قبل وبعد التنظيف.

In [ ]:
text_cols = ['customer_city', 'customer_state', 'product_category_name_english']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().str.strip()

**ببساطة:** بنوحّد شكل أسماء المدن والولايات والفئات، بإزالة المسافات وتحويل النص لحروف صغيرة، عشان ما نعتبرش نفس القيمة شكلين مختلفين.

In [ ]:
q99 = df['price'].quantile(0.99)
df_clean = df[df['price'] <= q99].copy()

**ببساطة:** بنحسب حد السعر عند 99% وبنستبعد الأسعار الأعلى منه من نسخة التحليل. ده تقليل لتأثير القيم المتطرفة، مش دليل إن كل سعر عالي غلط.

In [ ]:
df_clean.drop_duplicates(inplace=True)

**ببساطة:** بنشيل الصفوف المتطابقة بالكامل. لو الصفوف قلت بعد الخطوة، ده عدد التكرارات اللي اتشالت.


---

# Cleaning Impact

مقارنة حجم الداتا قبل التنظيف وبعده بشكل رقمي. المقارنة دي بتستخدم `df_original` المحفوظة قبل خطوات التنظيف و`df_clean` الناتجة من كود المشروع الحالي.


In [ ]:
before_rows, before_cols = df_original.shape
if "df_clean" in globals():
    after_rows, after_cols = df_clean.shape
else:
    after_rows, after_cols = df.shape
cleaning_impact = pd.DataFrame({
    "Stage": ["Before cleaning", "After cleaning"],
    "Rows": [before_rows, after_rows],
    "Columns": [before_cols, after_cols]
})
cleaning_impact["Rows Removed vs Original"] = [0, before_rows-after_rows]
cleaning_impact["Row Reduction %"] = [0, round((before_rows-after_rows)/before_rows*100, 2) if before_rows else 0]
display(cleaning_impact)
print(f"Rows removed: {before_rows-after_rows:,}")
print(f"Row reduction: {((before_rows-after_rows)/before_rows*100 if before_rows else 0):.2f}%")

**ببساطة:** الجدول بيقارن الحجم الأصلي بالحجم بعد التنظيف. Rows Removed هو عدد الصفوف اللي اختفت من نسخة التحليل، والنسبة بتوضح تأثير الخطوات على الحجم. لازم تراجع إن الحذف مبرر وما ضيّعش بيانات مفيدة.


In [ ]:
if "df_clean" in globals():
    print("Original duplicate rows:", int(df_original.duplicated().sum()))
    print("Clean dataset duplicate rows:", int(df_clean.duplicated().sum()))
    if "price" in df_original.columns and "price" in df_clean.columns:
        print("Original price rows:", int(df_original["price"].notna().sum()))
        print("Clean price rows:", int(df_clean["price"].notna().sum()))
        print("99th percentile price threshold used in existing cleaning step:", round(float(df_original["price"].quantile(0.99)), 2))
else:
    print("df_clean is not defined; run the original cleaning cells first.")

**ببساطة:** هنا بنراجع عدد التكرارات بعد التنظيف، وعدد الأسعار المتاحة قبل وبعده، وحد السعر عند 99% لو كان موجودًا. استبعاد الأسعار العالية يؤثر على التحليل ولا يعني بالضرورة إنها أخطاء.


# Final EDA Checklist

تمت إضافة أقسام مستقلة لـ `shape`, `info`, `describe`, Missing Values, Duplicate, Univariate, Bivariate, Multivariate, Boxplots, Correlation Matrix/Analysis, Sales & Categories, Delivery Analysis, Customer Reviews, Geographic Analysis, وCleaning Impact. شغّل الدفتر من أول خلية لآخر خلية علشان تتولد الأرقام والرسومات الفعلية.


# Decision Tree Model — Late Delivery (with Tree Drawing)

In [ ]:
# Build a clean classification dataset
required_cols = ['order_delivered_customer_date', 'order_estimated_delivery_date', 'order_purchase_timestamp', 'price', 'freight_value']
missing_required = [c for c in required_cols if c not in df_clean.columns]
print('Missing required columns:', missing_required)
if not missing_required:
    tree_data = df_clean.copy()
    for col in ['order_delivered_customer_date', 'order_estimated_delivery_date', 'order_purchase_timestamp']:
        tree_data[col] = pd.to_datetime(tree_data[col], errors='coerce')
    tree_data['is_late'] = (tree_data['order_delivered_customer_date'] > tree_data['order_estimated_delivery_date']).astype(int)
    tree_data['promised_delivery_days'] = (tree_data['order_estimated_delivery_date'] - tree_data['order_purchase_timestamp']).dt.days
    tree_data = tree_data.dropna(subset=['price', 'freight_value', 'promised_delivery_days', 'is_late'])
    features_tree = ['price', 'freight_value', 'promised_delivery_days']
    X_tree = tree_data[features_tree]
    y_tree = tree_data['is_late'].astype(int)
    print('Rows available for tree model:', len(tree_data))
    print('On-time orders (0):', int((y_tree == 0).sum()))
    print('Late orders (1):', int((y_tree == 1).sum()))
else:
    print('Tree model skipped until the required columns are available.')

**ببساطة:** `Rows available` عدد الطلبات اللي ينفع نستخدمها. `On-time (0)` الطلبات اللي وصلت في الموعد أو قبله، و`Late (1)` اللي وصلت بعد الموعد المتوقع. لو الفرق بين العددين كبير، الداتا غير متوازنة ولازم نهتم بـ Recall وF1 مش Accuracy بس.

In [ ]:
# Split data and train Decision Tree
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

if not missing_required and len(tree_data) > 20 and y_tree.nunique() == 2:
    X_train_tree, X_test_tree, y_train_tree, y_test_tree = train_test_split(
        X_tree, y_tree, test_size=0.2, random_state=42, stratify=y_tree
    )
    tree_model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, class_weight='balanced', random_state=42)
    tree_model.fit(X_train_tree, y_train_tree)
    y_pred_tree = tree_model.predict(X_test_tree)
    print('Accuracy:', round(accuracy_score(y_test_tree, y_pred_tree), 4))
    print('Precision:', round(precision_score(y_test_tree, y_pred_tree, zero_division=0), 4))
    print('Recall:', round(recall_score(y_test_tree, y_pred_tree, zero_division=0), 4))
    print('F1-score:', round(f1_score(y_test_tree, y_pred_tree, zero_division=0), 4))
    print('\nClassification report:')
    print(classification_report(y_test_tree, y_pred_tree, target_names=['On-time (0)', 'Late (1)'], zero_division=0))
else:
    print('Tree model not trained: check required columns, row count, and target classes.')

**ببساطة:** كل مقياس من 0 إلى 1؛ كل ما قرب من 1 كان أفضل، لكن لازم نقارن المقاييس مع بعض. في توقع التأخير، الـ Recall لفئة `Late (1)` مهم لأنه بيقولنا اكتشفنا كام نسبة من الطلبات المتأخرة فعلًا.

In [ ]:
# Confusion Matrix
if 'tree_model' in globals():
    cm_tree = confusion_matrix(y_test_tree, y_pred_tree, labels=[0, 1])
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_tree, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Predicted On-time', 'Predicted Late'],
                yticklabels=['Actual On-time', 'Actual Late'])
    plt.title('Decision Tree — Confusion Matrix')
    plt.xlabel('Predicted label')
    plt.ylabel('Actual label')
    plt.tight_layout()
    plt.show()
    print('TN (actual on-time predicted on-time):', cm_tree[0, 0])
    print('FP (actual on-time predicted late):', cm_tree[0, 1])
    print('FN (actual late predicted on-time):', cm_tree[1, 0])
    print('TP (actual late predicted late):', cm_tree[1, 1])

**ببساطة:** الأرقام على القطر الرئيسي هي التوقعات الصحيحة. `FN` هو أخطر رقم هنا غالبًا: طلب متأخر والمودل قال إنه هيوصل في المعاد. `TP` هو عدد الطلبات المتأخرة اللي المودل اكتشفها صح.

In [ ]:
# Draw the Decision Tree
if 'tree_model' in globals():
    plt.figure(figsize=(24, 12))
    plot_tree(tree_model, feature_names=features_tree,
              class_names=['On-time', 'Late'], filled=True,
              rounded=True, fontsize=9, proportion=True)
    plt.title('Decision Tree for Late Delivery Prediction (max_depth=4)')
    plt.tight_layout()
    plt.show()
    print('Tree depth:', tree_model.get_depth())
    print('Number of leaves:', tree_model.get_n_leaves())

**ببساطة:** كل مربع في الشجرة سؤال عن خاصية وحدّ معين. امشِ مع الفرع حسب إجابة السؤال لحد ما توصل لتوقع `On-time` أو `Late`. `Tree depth` عمق الشجرة، و`Number of leaves` عدد النهايات اللي بتطلع منها التوقعات. رسمنا الشجرة بعمق 4 عشان تفضل مقروءة.

In [ ]:
# Feature importance
if 'tree_model' in globals():
    importance_df = pd.DataFrame({'Feature': features_tree, 'Importance': tree_model.feature_importances_})
    importance_df = importance_df.sort_values('Importance', ascending=False)
    display(importance_df)
    sns.barplot(data=importance_df, x='Importance', y='Feature')
    plt.title('Decision Tree Feature Importance')
    plt.tight_layout()
    plt.show()

**ببساطة:** العمود صاحب أكبر `Importance` هو اللي الشجرة اعتمدت عليه أكثر في التقسيمات. لو القيمة `0` فالشجرة ما استخدمتش الخاصية دي في تقسيماتها. ده يوضح اعتماد المودل، مش إن الخاصية سبب التأخير وحدها.

# Run all cells — final checklist

In [ ]:
print('Notebook analysis sections are complete.')
print('Original project variables retained where possible: df, df_clean, rf_model, sales_forecast_model, xgb_churn_model, dt_model, lr_model, knn, multi_reg.')
print('Decision tree model:', 'trained' if 'tree_model' in globals() else 'not trained — inspect the messages above')

**ببساطة:** لو آخر خلية بتقول `trained` يبقى شجرة التأخير اتدرّبت. لو بتقول `not trained` راجع رسالة الأعمدة الناقصة أو عدد الصفوف اللي ظهرت في خلايا المودل.

In [ ]:
import pandas as pd
import plotly.express as px

**ببساطة:** هنا بنستورد مكتبات التحليل والرسوم. لو الخلية خلصت من غير Error يبقى الاستيراد نجح.

In [ ]:
df_clean['year_month'] = df_clean['order_purchase_timestamp'].dt.to_period('M').astype(str)

**ببساطة:** بنضيف عمود الشهر علشان نقدر نجمع المبيعات على مستوى كل شهر.

In [ ]:
monthly_sales = df_clean.groupby('year_month')['price'].sum().reset_index()

fig_trend = px.line(monthly_sales, 
                    x='year_month', 
                    y='price', 
                    title='Monthly Sales Trend Overview',
                    labels={'year_month': 'Timeline (Year-Month)', 'price': 'Total Revenue'},
                    markers=True)

fig_trend.update_layout(template='plotly_dark', hovermode='x unified')
fig_trend.show()

**ببساطة:** الرسم بيعرض إجمالي أسعار المنتجات شهريًا. أعلى نقطة هي الشهر صاحب أعلى إجمالي مبيعات حسب العمود المستخدم.

In [ ]:
# --- Dashboard 2: Top 10 Product Categories ---
top_categories = df_clean.groupby('product_category_name_english')['price'].sum().reset_index()
top_categories = top_categories.sort_values(by='price', ascending=False).head(10)

fig_cats = px.bar(top_categories, 
                  x='price', 
                  y='product_category_name_english', 
                  orientation='h',
                  title='Top 10 Performing Product Categories',
                  labels={'price': 'Total Revenue', 'product_category_name_english': 'Product Category'},
                  color='price',
                  color_continuous_scale='Blues')

fig_cats.update_layout(template='plotly_dark', yaxis={'categoryorder':'total ascending'})
fig_cats.show()

**ببساطة:** دي أعلى 10 فئات حسب إجمالي قيمة المنتجات. الأرقام بتوضح الفئات الأعلى قيمة في الداتا.

In [ ]:

layout_style = 'plotly_dark'

if 'payment_type' in df_clean.columns:
    pay_dist = df_clean['payment_type'].value_counts().reset_index()
    pay_dist.columns = ['payment_type', 'count']
    fig3 = px.pie(pay_dist, names='payment_type', values='count', hole=0.4,
                  title='3. Customer Payment Methods Preference')
    fig3.update_layout(template=layout_style)
    fig3.show()

**ببساطة:** الرسم بيعرض توزيع طرق الدفع. أكبر شريحة أو رقم بيوضح طريقة الدفع الأكثر ظهورًا في البيانات.

In [ ]:
df_clean['day_of_week'] = df_clean['order_purchase_timestamp'].dt.day_name()
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_sales = df_clean.groupby('day_of_week')['order_id'].count().reset_index()

fig4 = px.bar(day_sales, x='day_of_week', y='order_id', 
              title='4. Order Volume by Day of the Week',
              category_orders={'day_of_week': days_order}, 
              color='order_id', color_continuous_scale='Purples')

fig4.update_layout(template=layout_style)
fig4.show()

**ببساطة:** بنقارن النشاط حسب أيام الأسبوع. أعلى قيمة في الناتج معناها اليوم الأكثر نشاطًا حسب المقياس المستخدم.

In [ ]:
if 'customer_state' in df_clean.columns:
    state_sales = df_clean.groupby('customer_state')['price'].sum().reset_index()
    state_sales = state_sales.sort_values(by='price', ascending=False).head(10)
    fig5 = px.bar(state_sales, x='customer_state', y='price',
                  title='5. Top 10 Customer States by Total Revenue', color='price', color_continuous_scale='Teal')
    fig5.update_layout(template=layout_style)
    fig5.show()

**ببساطة:** الرسم بيرتب الولايات حسب المبيعات. ده ترتيب بالقيمة مش بالضرورة بعدد العملاء.

In [ ]:
df_clean['delivery_days'] = (df_clean['order_delivered_customer_date'] - df_clean['order_purchase_timestamp']).dt.days
fig6 = px.histogram(df_clean[df_clean['delivery_days'] <= 40], x='delivery_days', nbins=40,
                    title='6. Delivery Time Distribution (in Days)')
fig6.update_layout(template=layout_style, xaxis_title='Days to Deliver', yaxis_title='Number of Orders')
fig6.show()

**ببساطة:** توزيع مدة التوصيل بيوضح الطلبات غالبًا بتوصل بعد كام يوم. الذيل الطويل معناه فيه طلبات استغرقت وقتًا أطول من المعتاد.

In [ ]:
top_cats = df_clean.groupby('product_category_name_english')['price'].sum().reset_index()
top_cats = top_cats.sort_values(by='price', ascending=False).head(10)
if 'review_score' in df_clean.columns:
    top_cat_names = top_cats['product_category_name_english'].tolist()
    reviews = df_clean[df_clean['product_category_name_english'].isin(top_cat_names)]
    avg_reviews = reviews.groupby('product_category_name_english')['review_score'].mean().reset_index()
    fig7 = px.bar(avg_reviews, x='review_score', y='product_category_name_english', orientation='h',
                  title='7. Average Customer Satisfaction (Top Categories)', range_x=[0, 5])
    fig7.update_layout(template=layout_style, yaxis={'categoryorder':'total ascending'})
    fig7.show()

**ببساطة:** بنقارن الفئات الأعلى مبيعًا مع تقييماتها إن كان عمود التقييم موجودًا. القيم الأعلى للتقييم معناها رضا أعلى في المتوسط.

In [ ]:
cat_metrics = df_clean.groupby('product_category_name_english').agg(
    avg_price=('price', 'mean'), avg_freight=('freight_value', 'mean'), total_orders=('order_id', 'count')
).reset_index()
cat_metrics = cat_metrics[cat_metrics['total_orders'] > 50] # Filter small categories
fig8 = px.scatter(cat_metrics, x='avg_price', y='avg_freight', size='total_orders', 
                  hover_name='product_category_name_english',
                  title='8. Relationship: Avg Item Price vs. Avg Freight Value per Category',
                  color='avg_price', color_continuous_scale='Magma')
fig8.update_layout(template=layout_style)
fig8.show()

**ببساطة:** الجدول بيقارن متوسط السعر ومتوسط الشحن وعدد الطلبات لكل فئة. عدد الطلبات الأكبر يعني بيانات أكثر للفئة.

In [ ]:
%pip install --user xgboost scikit-learn joblib

**ببساطة:** دي خلية تثبيت المكتبات المطلوبة. لو ظهر خطأ تثبيت، ثبّت المكتبة الناقصة في نفس بيئة الـ Python المستخدمة للـ Notebook.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
# 2. Data Preprocessing & Dimensionality Reduction (PCA)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.decomposition import PCA


from sklearn.linear_model import LinearRegression

from sklearn.svm import SVC, SVR

from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.naive_bayes import GaussianNB

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor

from sklearn.cluster import KMeans, DBSCAN

from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             mean_squared_error, mean_absolute_error, r2_score)

print("All Machine Learning libraries imported successfully!")

**ببساطة:** بنستورد أدوات تقسيم البيانات وتجهيزها والموديلات. رسالة النجاح في آخر الخلية معناها المكتبات اتعمل لها import.

In [ ]:
latest_date = df_clean['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = df_clean.groupby('customer_id').agg({
    'order_purchase_timestamp': lambda x: (latest_date - x.max()).days,
    'order_id': 'nunique',
    'price': 'sum'
}).reset_index()

rfm.columns = ['customer_id', 'Recency', 'Frequency', 'Monetary']

rfm = rfm[(rfm['Monetary'] <= rfm['Monetary'].quantile(0.99)) & 
          (rfm['Frequency'] <= rfm['Frequency'].quantile(0.99))]

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

cluster_mapping = {
    0: 'Potential/New Customers',
    1: 'At-Risk/Churned Customers',
    2: 'Loyal Customers',
    3: 'VIP/High-Spenders'
}
rfm['Customer_Segment'] = rfm['Cluster'].map(cluster_mapping)

print("Clustering completed successfully!")
print(rfm['Customer_Segment'].value_counts())

layout_style = 'plotly_dark'

fig_kmeans = px.scatter_3d(rfm, 
                           x='Recency', 
                           y='Frequency', 
                           z='Monetary',
                           color='Customer_Segment',
                           title='Customer Segmentation (3D K-Means Clustering)',
                           opacity=0.7,
                           color_discrete_sequence=px.colors.qualitative.Pastel)

fig_kmeans.update_layout(template=layout_style, 
                         scene=dict(xaxis_title='Recency (Days)',
                                    yaxis_title='Frequency (Orders)',
                                    zaxis_title='Monetary (Revenue)'))
fig_kmeans.show()

**ببساطة:** بنقسم العملاء لأربع مجموعات بناءً على آخر شراء وعدد الطلبات وإجمالي الإنفاق. أعداد كل مجموعة تظهر في الجدول والرسم؛ راجع خصائص كل مجموعة قبل اعتماد الاسم.

In [ ]:
df_clean['is_late'] = (df_clean['order_delivered_customer_date'] > df_clean['order_estimated_delivery_date']).astype(int)

df_clean['promised_delivery_days'] = (df_clean['order_estimated_delivery_date'] - df_clean['order_purchase_timestamp']).dt.days

features = ['price', 'freight_value', 'promised_delivery_days']
X = df_clean[features].dropna()
y = df_clean.loc[X.index, 'is_late']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training the Random Forest model (this might take a few seconds)...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("\nModel Evaluation (Classification Report):")
print(classification_report(y_test, y_pred))

layout_style = 'plotly_dark'
cm = confusion_matrix(y_test, y_pred)

fig_cm = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                   labels=dict(x="Predicted Label", y="True Label", color="Count"),
                   x=['On Time (0)', 'Late (1)'], y=['On Time (0)', 'Late (1)'],
                   title='Confusion Matrix: Late Delivery Prediction (Random Forest)')

fig_cm.update_layout(template=layout_style)
fig_cm.show()

**ببساطة:** المودل بيتوقع هل الطلب متأخر (1) ولا لأ (0). Precision: دقة الطلبات التي توقعنا تأخرها، وRecall: نسبة الطلبات المتأخرة التي اكتشفها. راجع مصفوفة الالتباس كمان.

In [ ]:
import plotly.graph_objects as go

df_clean['order_purchase_timestamp'] = pd.to_datetime(df_clean['order_purchase_timestamp'])
df_clean['purchase_date'] = df_clean['order_purchase_timestamp'].dt.date

daily_sales = df_clean.groupby('purchase_date')['price'].sum().reset_index()
daily_sales.columns = ['date', 'total_sales']
daily_sales['date'] = pd.to_datetime(daily_sales['date'])

daily_sales = daily_sales.sort_values('date')

daily_sales['day_of_week'] = daily_sales['date'].dt.dayofweek
daily_sales['day_of_month'] = daily_sales['date'].dt.day
daily_sales['month'] = daily_sales['date'].dt.month
daily_sales['year'] = daily_sales['date'].dt.year

X = daily_sales[['day_of_week', 'day_of_month', 'month', 'year']]
y = daily_sales['total_sales']

split_idx = int(len(daily_sales) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test = daily_sales['date'].iloc[split_idx:]

print("Training XGBoost Regressor for Sales Forecasting...")
sales_forecast_model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
sales_forecast_model.fit(X_train, y_train)

y_pred = sales_forecast_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
print("\nModel Evaluation:")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")

layout_style = 'plotly_dark'
fig_forecast = go.Figure()

fig_forecast.add_trace(go.Scatter(x=dates_test, y=y_test, mode='lines', 
                                  name='Actual Sales', line=dict(color='cyan')))
fig_forecast.add_trace(go.Scatter(x=dates_test, y=y_pred, mode='lines', 
                                  name='Predicted Sales', line=dict(color='orange', dash='dash')))

fig_forecast.update_layout(title='Sales Forecasting: Actual vs Predicted (Testing Period)',
                           xaxis_title='Date',
                           yaxis_title='Total Sales Revenue',
                           template=layout_style,
                           hovermode="x unified")
fig_forecast.show()

**ببساطة:** بنقارن المبيعات اليومية الحقيقية بالتوقعات. MAE هو متوسط مقدار الخطأ بنفس وحدة المبيعات؛ الأقل أفضل عادة. الخطان المتقاربين يعني توقع أقرب للحقيقة.

In [ ]:
df_clean['delivery_days'] = (pd.to_datetime(df_clean['order_delivered_customer_date']) - 
                             pd.to_datetime(df_clean['order_purchase_timestamp'])).dt.days

df_clean['estimated_days'] = (pd.to_datetime(df_clean['order_estimated_delivery_date']) - 
                              pd.to_datetime(df_clean['order_purchase_timestamp'])).dt.days

features = ['price', 'freight_value', 'delivery_days', 'estimated_days']

df_subset = df_clean.dropna(subset=features + ['review_score'])

X = df_subset[features]
y = df_subset['review_score'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Decision Tree Classifier...")
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced')
dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)

print("\nClassification Report (1 to 5 Stars):")
print(classification_report(y_test, y_pred, zero_division=0))

layout_style = 'plotly_dark'
importance = dt_model.feature_importances_

fig_imp = px.bar(x=importance, y=features, orientation='h',
                 title='Feature Importance: What drives customer satisfaction?',
                 labels={'x': 'Impact Level (Importance)', 'y': 'Store Metric'},
                 color=importance, color_continuous_scale='Reds')

fig_imp.update_layout(template=layout_style, yaxis={'categoryorder':'total ascending'})
fig_imp.show()

**ببساطة:** توزيع مدة التوصيل بيوضح الطلبات غالبًا بتوصل بعد كام يوم. الذيل الطويل معناه فيه طلبات استغرقت وقتًا أطول من المعتاد.

In [ ]:
latest_date = df_clean['order_purchase_timestamp'].max()

churn_data = df_clean.groupby('customer_id').agg(
    Recency=('order_purchase_timestamp', lambda x: (latest_date - x.max()).days),
    Monetary=('price', 'sum'),
    Avg_Review=('review_score', 'mean'),
    Avg_Freight=('freight_value', 'mean'),
    Avg_Delivery=('delivery_days', 'mean')
).reset_index()

churn_data['Churn'] = (churn_data['Recency'] > 180).astype(int)

features = ['Monetary', 'Avg_Review', 'Avg_Freight', 'Avg_Delivery']
X = churn_data[features].dropna()
y = churn_data.loc[X.index, 'Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training XGBoost Classifier...")
xgb_churn_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, eval_metric='logloss')
xgb_churn_model.fit(X_train, y_train)

y_pred = xgb_churn_model.predict(X_test)

print("\nModel Evaluation (Churn Prediction - 0: Active, 1: Churned):")
print(classification_report(y_test, y_pred, zero_division=0))

layout_style = 'plotly_dark'
importance = xgb_churn_model.feature_importances_

fig_churn = px.bar(x=importance, y=features, orientation='h',
                   title='Factors Influencing Customer Churn (XGBoost)',
                   labels={'x': 'Impact Level (Importance)', 'y': 'Customer Metric'},
                   color=importance, color_continuous_scale='Viridis')

fig_churn.update_layout(template=layout_style, yaxis={'categoryorder':'total ascending'})
fig_churn.show()

**ببساطة:** هنا تعريف churn افتراضي: العميل اللي آخر شراء له أقدم من 180 يوم بنعتبره غير نشط. التقرير يوضح جودة التصنيف، لكن التعريف مش حقيقة مؤكدة عن العميل.

In [ ]:
clv_data = df_clean.groupby('customer_id').agg(
    Frequency=('order_id', 'nunique'),
    Avg_Freight=('freight_value', 'mean'),
    Avg_Delivery=('delivery_days', 'mean'),
    Monetary=('price', 'sum')
).reset_index()

features = ['Frequency', 'Avg_Freight', 'Avg_Delivery']
X = clv_data[features].dropna()
y = clv_data.loc[X.index, 'Monetary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Linear Regression Model...")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"\nModel Evaluation:")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"R-squared Score (R2): {r2:.2f}")

layout_style = 'plotly_dark'

fig_clv = px.scatter(x=y_test, y=y_pred, 
                     labels={'x': 'Actual Customer Value ($)', 'y': 'Predicted Customer Value ($)'},
                     title='Customer Lifetime Value Prediction (Linear Regression)',
                     opacity=0.6, color_discrete_sequence=['cyan'])

max_val = max(y_test.max(), y_pred.max())
fig_clv.add_trace(go.Scatter(x=[0, max_val], y=[0, max_val], mode='lines', 
                             name='Perfect Prediction', line=dict(color='orange', dash='dash')))

fig_clv.update_layout(template=layout_style, hovermode="closest")
fig_clv.show()

**ببساطة:** بنحاول نتوقع قيمة العميل من عدد الطلبات ومتوسط الشحن والتوصيل. MAE هو متوسط الخطأ، وR² الأقرب إلى 1 غالبًا أفضل؛ القريب من صفر أو السالب معناه أداء ضعيف.

In [ ]:
from sklearn.neighbors import NearestNeighbors

category_data = df_clean.groupby('product_category_name_english').agg(
    Avg_Price=('price', 'mean'),
    Avg_Freight=('freight_value', 'mean'),
    Total_Orders=('order_id', 'count'),
    Avg_Review=('review_score', 'mean')
).reset_index()

category_data = category_data[category_data['Total_Orders'] > 50].reset_index(drop=True)

features = ['Avg_Price', 'Avg_Freight', 'Total_Orders', 'Avg_Review']
X_cats = category_data[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cats)

print("Training KNN Model for Recommendations...")
knn = NearestNeighbors(n_neighbors=4, metric='euclidean')
knn.fit(X_scaled)

distances, indices = knn.kneighbors(X_scaled)

test_category = 'health_beauty'
if test_category in category_data['product_category_name_english'].values:
    cat_idx = category_data[category_data['product_category_name_english'] == test_category].index[0]
    
    neighbor_indices = indices[cat_idx][1:]
    recommended_cats = category_data.iloc[neighbor_indices]['product_category_name_english'].tolist()
    
    print(f"\n--- [Recommendation Engine Demo] ---")
    print(f"If a customer is viewing: '{test_category.upper()}'")
    print(f"The KNN model suggests cross-selling: {recommended_cats}")
    print(f"------------------------------------\n")

layout_style = 'plotly_dark'
top_n = 20 
fig_knn = go.Figure()

sample_cats = category_data.sort_values('Total_Orders', ascending=False).head(top_n)

fig_knn.add_trace(go.Scatter(
    x=sample_cats['Avg_Price'], 
    y=sample_cats['Avg_Review'],
    mode='markers+text',
    text=sample_cats['product_category_name_english'],
    textposition="top center",
    marker=dict(size=sample_cats['Total_Orders']/200, color='cyan', opacity=0.7),
    name='Product Categories'
))

fig_knn.update_layout(
    title='Product Category Similarity Matrix (KNN Feature Space)',
    xaxis_title='Average Price ($)',
    yaxis_title='Average Review Score (Stars)',
    template=layout_style,
    hovermode="closest"
)
fig_knn.show()

**ببساطة:** KNN بيقترح فئات منتجات متشابهة حسب السعر والشحن وعدد الطلبات والتقييم. الفئات المقترحة بتظهر لو الفئة التجريبية موجودة في البيانات.

In [ ]:
physical_features = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

available_features = [col for col in physical_features if col in df_clean.columns]

if len(available_features) > 1:
    print(f"Applying PCA on physical features: {available_features}")
    df_freight = df_clean.dropna(subset=available_features + ['freight_value', 'price']).copy()
    
    scaler = StandardScaler()
    scaled_dims = scaler.fit_transform(df_freight[available_features])
    
    pca = PCA(n_components=1, random_state=42)
    df_freight['Size_Index'] = pca.fit_transform(scaled_dims)
    
    X = df_freight[['Size_Index', 'price']]
else:
    print("Physical features missing. Using 'price' and order counts as proxy features.")
    df_freight = df_clean.dropna(subset=['price', 'freight_value']).copy()
    X = df_freight[['price']] 

y = df_freight['freight_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Multiple Regression Model...")
multi_reg = LinearRegression()
multi_reg.fit(X_train, y_train)

y_pred = multi_reg.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nModel Evaluation (Predicting Freight Value):")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"R-squared Score (R2): {r2:.2f}")

layout_style = 'plotly_dark'

plot_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred}).sample(n=min(1000, len(y_test)), random_state=42)

fig_freight = px.scatter(plot_df, x='Actual', y='Predicted', 
                         title='8. Freight Value Prediction (PCA + Multi Regression)',
                         labels={'Actual': 'Actual Freight Cost ($)', 'Predicted': 'Predicted Freight Cost ($)'},
                         opacity=0.6, color_discrete_sequence=['#00CC96'])

max_val = max(plot_df['Actual'].max(), plot_df['Predicted'].max())
fig_freight.add_shape(type="line", x0=0, y0=0, x1=max_val, y1=max_val,
                      line=dict(color="orange", width=2, dash="dash"))

fig_freight.update_layout(template=layout_style, hovermode="closest")
fig_freight.show()

**ببساطة:** PCA بيختصر أبعاد المنتج الفيزيائية في مؤشر واحد، وبعدها المودل بيتوقع تكلفة الشحن. MAE الأقل أفضل عادة، وR² بيوضح مدى تفسير النموذج لاختلاف القيم.

In [ ]:
import joblib

# Exporting models to disk
joblib.dump(rf_model, 'rf_late_delivery_model.pkl')
joblib.dump(xgb_churn_model, 'xgb_churn_model.pkl')

print("Models saved successfully!")

**ببساطة:** بص على الناتج اللي ظهر فوق. لو فيه جدول، ركّز على عدد الصفوف والأعمدة والقيم؛ ولو فيه رسم، قارن أعلى وأقل القيم. الأرقام الفعلية هتظهر بعد تشغيل الخلية على الداتا.

In [ ]:
df_clean.to_csv('cleaned_data.csv', index=False)

**ببساطة:** بص على الناتج اللي ظهر فوق. لو فيه جدول، ركّز على عدد الصفوف والأعمدة والقيم؛ ولو فيه رسم، قارن أعلى وأقل القيم. الأرقام الفعلية هتظهر بعد تشغيل الخلية على الداتا.

**ببساطة:** بص على الناتج اللي ظهر فوق. لو فيه جدول، ركّز على عدد الصفوف والأعمدة والقيم؛ ولو فيه رسم، قارن أعلى وأقل القيم. الأرقام الفعلية هتظهر بعد تشغيل الخلية على الداتا.